# Итоги лабораторных 1–5

Сравниваем алгоритмы на одних и тех же тестовых выборках каждой задачи. Не переобучаем модели: читаем по восемь строк из пяти CSV. Порядок экспериментов и метрики описаны в [протоколе оценки](../docs/data-protocol.md).

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
paths = [root / "results" / f"lab{lab:02d}.csv" for lab in range(1, 6)]
results = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
print(f"Всего результатов: {len(results)}")

Всего результатов: 40


## Классификация: Spambase

Сортируем 20 вариантов по F1-macro (выше — лучше), затем по accuracy. Округление применяется только при отображении, после сортировки.

In [2]:
classification = (
    results.loc[results["task"] == "classification", ["lab", "algorithm", "variant", "f1_macro", "accuracy"]]
    .sort_values(["f1_macro", "accuracy"], ascending=False)
    .reset_index(drop=True)
)
display(classification.round(5))

,lab,algorithm,variant,f1_macro,accuracy
0,4,RandomForest,custom_improved,0.95277,0.95482
1,4,RandomForest,sklearn_improved,0.95267,0.95482
2,4,RandomForest,custom_base,0.94658,0.94887
3,4,RandomForest,sklearn_base,0.94421,0.94649
4,5,GradientBoosting,sklearn_base,0.94277,0.94530
5,5,GradientBoosting,sklearn_improved,0.94277,0.94530
6,3,DecisionTree,sklearn_improved,0.92768,0.93103
7,3,DecisionTree,custom_improved,0.92640,0.92985
8,2,LogisticRegression,sklearn_improved,0.92558,0.92866
9,2,LogisticRegression,custom_base,0.92182,0.92509


## Регрессия: SML2010

Сортируем 20 вариантов по MAE (ниже — лучше), затем по RMSE. Обе метрики измеряются в °C.

In [3]:
regression = (
    results.loc[results["task"] == "regression", ["lab", "algorithm", "variant", "mae", "rmse"]]
    .sort_values(["mae", "rmse"])
    .reset_index(drop=True)
)
display(regression.round(5))

,lab,algorithm,variant,mae,rmse
0,2,LinearRegression,sklearn_base,0.02073,0.02841
1,2,LinearRegression,sklearn_improved,0.02073,0.02841
2,2,LinearRegression,custom_base,0.02073,0.02841
3,2,LinearRegression,custom_improved,0.02073,0.02841
4,4,RandomForest,custom_base,0.28516,0.67029
5,4,RandomForest,custom_improved,0.28516,0.67029
6,4,RandomForest,sklearn_base,0.30212,0.70375
7,4,RandomForest,sklearn_improved,0.30212,0.70375
8,5,GradientBoosting,sklearn_base,0.30969,0.68567
9,5,GradientBoosting,sklearn_improved,0.30969,0.68567


## Что дали улучшения

Для каждой пары «алгоритм и реализация» сравниваем конфигурацию после подбора с исходной. Для F1-macro показываем `улучшенный − базовый`, для MAE — `базовый − улучшенный`. В обеих колонках положительное число означает улучшение.

In [4]:
def gains(table, metric, direction, label):
    pair = table.pivot(index="algorithm", columns="variant", values=metric)
    return pd.DataFrame({
        f"{label}, sklearn": direction * (pair["sklearn_improved"] - pair["sklearn_base"]),
        f"{label}, NumPy": direction * (pair["custom_improved"] - pair["custom_base"]),
    }).round(5)


display(gains(classification, "f1_macro", 1, "Прирост F1-macro"))
display(gains(regression, "mae", -1, "Снижение MAE, °C"))

,"Прирост F1-macro, sklearn","Прирост F1-macro, NumPy"
algorithm,,
DecisionTree,0.00843,0.01203
GradientBoosting,0.00000,0.00000
KNN,0.13858,0.13858
LogisticRegression,0.00496,-0.00248
RandomForest,0.00846,0.00619


,"Снижение MAE, °C, sklearn","Снижение MAE, °C, NumPy"
algorithm,,
DecisionTree,0.01525,0.00032
GradientBoosting,-0.00000,-0.00000
KNN,1.80454,1.80454
LinearRegression,-0.00000,-0.00000
RandomForest,-0.00000,-0.00000


## Выводы

- **Spambase.** Лучший F1-macro у случайного леса из 60 деревьев: 0,95277 у NumPy и 0,95267 у sklearn при одинаковой accuracy. Разница между реализациями мала на этом тесте. Самый большой прирост после подбора получил KNN: стандартизация подняла F1-macro с 0,77433 до 0,91291.
- **SML2010.** Линейная регрессия лидирует с MAE 0,02073 °C у обеих реализаций. Она единственная из пяти семейств обошла прогноз по текущей температуре (MAE 0,11636 °C). У KNN MAE заметно снизилась после масштабирования, но осталась хуже этого ориентира.
- **Сравнение реализаций.** KNN и линейная регрессия совпали со sklearn по сохранённым метрикам. Лес показал близкое качество. Наиболее заметный разрыв в классификации — у градиентного бустинга: 0,91455 у NumPy против 0,94277 у sklearn. В учебной версии нет отдельного ньютоновского обновления листьев.

Это сравнение относится к двум фиксированным тестам. Различия в несколько десятитысячных не доказывают, что одна реализация устойчиво лучше другой. Мы не сравниваем F1-macro и MAE между задачами.